# ProxyTrace 04. 주장 추출 (GPT)

03 노트북이 나눈 **주주의 말 / 회사의 말**에서 검증할 주장을 뽑아요.

| 필드 | 뜻 |
| --- | --- |
| `quote` | 위임장 원문 그대로 (LLM이 바꾸지 않게) |
| `claim_ko` | 한국어 한 문장 요약 |
| `kind` | 사실(fact) / 수치(number) / 의견·약속(opinion_commitment) |
| `checkable` | 외부 자료로 확인할 수 있는가 |
| `topic` | 주제 (ESG 태깅 때 쓸 분류) |
| `page_no` | 원문이 실제로 있는 쪽 (코드가 직접 찾아서 확인) |
| `in_board_summary` | 회사가 요약 bullet에 넣은 핵심 주장인가 |

결과: `data/claims/{TICKER}/{제출일}_claims.json`, `.csv`

In [1]:
!pip install openai -q

In [3]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [12]:
!pip install pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 18.5 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 16.1 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


## 1. 설정
- `TEST_NUMBERS = [4]`: 처음엔 **4번 안건 하나만** 돌려서 결과를 보고 프롬프트를 고쳐요.
- 결과가 괜찮으면 `None`으로 바꾸고 전체 10개를 돌려요.
- API 키는 입력창에 넣어요. 코드에 저장되지 않아요.

In [ ]:
import csv
import json
import os
import re
from getpass import getpass
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv    
load_dotenv(".env") 

# ▶ 01~03 노트북과 같은 티커
TICKER = "META"

# ▶ 모델: 주장 추출은 작은 모델로 충분해요. 쓸 수 있는 모델 이름으로 바꿔도 돼요.
MODEL = "gpt-4o-mini"

# ▶ 처음엔 4번 안건 하나만 → 결과 확인 후 None(전체 10개)으로 바꾸기
TEST_NUMBERS = None    # None = 전체 10개

# API 키: 환경변수 OPENAI_API_KEY가 없으면 입력창이 떠요 (코드에 저장 안 됨)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ")
client = OpenAI()

T = TICKER.lower()
manifest = json.loads((Path("data/raw") / T / "manifest.json").read_text(encoding="utf-8"))
FILING = manifest["proxies"][0]["filingDate"]
parsed = json.loads((Path("data/parsed") / T / f"{FILING}_proposals.json").read_text(encoding="utf-8"))
pages = json.loads((Path("data/clean") / T / f"{FILING}_pages.json").read_text(encoding="utf-8"))["pages"]

CLAIM_DIR = Path("data/claims") / T
CLAIM_DIR.mkdir(parents=True, exist_ok=True)
print(f"{manifest['company']} {FILING}: 주주제안 {len(parsed['proposals'])}개")

Meta Platforms, Inc. 2026-04-16: 주주제안 10개


## 2. 출력 형식 + 지시문
LLM이 항상 같은 형식(JSON)으로 답하게 강제해요. 지시문은 영어가 정확도가 더 높아서 영어로 썼어요.

In [ ]:
TOPICS = [
    "voting_governance",      # 투표·의결권·지배구조·이사회
    "compensation",           # 경영진 보수
    "privacy_ai_data",        # 개인정보·AI 데이터
    "human_rights_content",   # 인권·콘텐츠 정책·혐오
    "child_safety",           # 아동 안전
    "climate_energy",         # 기후·에너지·데이터센터
    "workforce_immigration",  # 인력·고용·이민
    "other",
]

CLAIM_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["claims"],
    "properties": {
        "claims": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["quote", "claim_ko", "kind", "checkable", "topic", "time_ref"],
                "properties": {
                    "quote":     {"type": "string", "description": "원문에서 그대로 복사한 문장 (수정 금지)"},
                    "claim_ko":  {"type": "string", "description": "주장을 한국어 한 문장으로"},
                    "kind":      {"type": "string", "enum": ["fact", "number", "opinion_commitment", "request", "allegation"]},
                    "checkable": {"type": "boolean", "description": "외부 자료로 사실 여부를 확인할 수 있는가"},
                    "topic":     {"type": "string", "enum": TOPICS},
                    "time_ref":  {"type": "string", "description": "주장이 가리키는 시점 (예: '2025 Annual Meeting'), 없으면 빈 문자열"},
                },
            },
        }
    },
}

SYSTEM = """You extract verifiable claims from a U.S. proxy statement for an investor fact-checking tool.

Rules:
- Split the text into atomic claims: one checkable statement per claim. If one sentence contains two facts, make two claims that share the same quote.
- "quote" MUST be copied exactly from the text (same words, same punctuation). Never paraphrase inside "quote".
- kind:
  - fact: a statement about something that exists or happened (e.g., "we publish X", "shareholders approved Y in 2025").
  - number: a statement whose core is a figure, percentage, count or amount.
  - opinion_commitment: beliefs, values, intentions or promises ("we believe", "we are committed to", "is in the best interests").
  - request: what the proponent asks the company or shareholders to do ("Shareholders ask Meta to...", "We urge shareholders to vote FOR...").
  - allegation: an accusation of wrongdoing or misconduct, often hedged ("it is believed that ... used improper means").
- Hidden facts: if a request, opinion or allegation contains a past fact or a figure, ALSO extract that fact as its own fact/number claim (same quote).
  Example: "the same vote that was on the 2025 ballot, to be on each annual ballot" → request + fact "a say-on-pay vote was on the 2025 ballot".
- No repeats: if the same fact appears more than once in the text (for example in a bullet summary and again in the body), extract it ONCE, using the quote from its FIRST appearance.
- checkable = true only if an outside document (SEC filing, company report or webpage, government data) could confirm or contradict it.
  opinion_commitment and request are always false. allegation is false unless it names a specific, documentable event.
- Skip boilerplate (voting instructions, footnote URLs, "the board recommends a vote AGAINST").
- Write claim_ko in Korean.
"""

## 3. 함수 정의

In [17]:
SPEAKERS = {"shareholder": ("proponent_text", "S"), "company": ("board_text", "C")}


def ws(s: str) -> str:
    """공백·따옴표 차이를 무시하고 비교하기 위한 정리"""
    s = s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    return re.sub(r"\s+", " ", s).strip().lower()


PAGE_TEXT = [(p["page_no"], ws(p["text"])) for p in pages]


def find_page(quote: str):
    """인용문이 실제로 위임장 몇 쪽에 있는지 찾기. 못 찾으면 None (= LLM이 문장을 바꿨을 가능성)"""
    q = ws(quote)
    for key in (q, q[:80], q[:40]):
        if len(key) < 15:
            break
        for no, txt in PAGE_TEXT:
            if key in txt:
                return no
    return None


def lead_proponent(s: str) -> str:
    """'lead filer A and co-filers B, C' → A (대표 제안자만)"""
    if not s:
        return s
    m = re.search(r"(?:co-)?lead filer\s+(.+?)(?:,?\s+and\s+(?:co-lead filer|co-filers?)|,\s*co-lead filer|$)", s, re.I)
    name = m.group(1) if m else s
    return re.sub(r"^the\s+", "", name.strip(" ,."), flags=re.I)


def extract(text: str, title: str, speaker: str) -> tuple[list[dict], dict]:
    who = "the shareholder proponent" if speaker == "shareholder" else "the company's board of directors"
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Proposal: {title}\\nSpeaker: {who}\\n\\nText:\\n{text}"},
        ],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "claims", "schema": CLAIM_SCHEMA, "strict": True}},
    )
    data = json.loads(resp.choices[0].message.content)
    u = resp.usage
    return data["claims"], {"input": u.prompt_tokens, "output": u.completion_tokens}


def run(proposal: dict) -> tuple[list[dict], dict]:
    out, usage = [], {"input": 0, "output": 0}
    summary = [ws(b) for b in proposal["board_summary"]]
    for speaker, (field, code) in SPEAKERS.items():
        claims, u = extract(proposal[field], proposal["title"], speaker)
        usage = {k: usage[k] + u[k] for k in usage}
        for i, c in enumerate(claims, 1):
            page = find_page(c["quote"])
            out.append({
                "claim_id": f"P{proposal['number']:02d}-{code}{i:02d}",
                "proposal": proposal["number"],
                "proposal_topic": proposal["topic"],
                "speaker": speaker,
                "proponent": lead_proponent(proposal["proponent"]),
                **c,
                "page_no": page,
                "quote_found": page is not None,
                # 회사가 요약 bullet에 직접 넣은 주장인가 (= 회사가 강조한 핵심 주장)
                "in_board_summary": speaker == "company" and any(ws(c["quote"])[:60] in s or s[:60] in ws(c["quote"]) for s in summary),
            })
    return out, usage

## 4. 실행

In [19]:
targets = [p for p in parsed["proposals"] if TEST_NUMBERS is None or p["number"] in TEST_NUMBERS]
all_claims, total = [], {"input": 0, "output": 0}
for p in targets:
    claims, u = run(p)
    all_claims += claims
    total = {k: total[k] + u[k] for k in total}
    print(f"[{p['number']:>2}] {p['topic'][:40]:40} 주장 {len(claims):>3}개  (토큰 in {u['input']:,} / out {u['output']:,})")
print(f"\n합계: 주장 {len(all_claims)}개, 토큰 in {total['input']:,} / out {total['output']:,}")

[ 4] Annual Vote Regarding Executive Pay      주장  15개  (토큰 in 1,959 / out 1,314)

합계: 주장 15개, 토큰 in 1,959 / out 1,314


## 5. 결과 확인 ⚠ 중요
- **`quote_found`가 False인 주장**: LLM이 원문을 바꿔 쓴 것 → 프롬프트를 고치거나 버려야 해요
- `kind`, `checkable`이 상식적인지
- 같은 내용이 너무 잘게 쪼개지거나, 반대로 여러 개가 한 주장에 뭉쳐 있지 않은지

In [ ]:
for c in all_claims:
    flag = "" if c["quote_found"] else "  ⚠ 원문에서 못 찾음"
    star = "★" if c["in_board_summary"] else " "
    chk = "검증가능" if c["checkable"] else "검증불가"
    print(f"{c['claim_id']} {star} p.{c['page_no']} | {c['speaker']:11} | {c['kind']:18} | {chk} | {c['topic']}{flag}")
    print(f"    {c['claim_ko']}")
    print(f"    \"{c['quote'][:150]}\"")

P04-S01   p.70 | shareholder | opinion_commitment | 검증불가 | voting_governance
    주주들은 메타가 '연례 임금 투표'라는 이름으로 메타의 임원 보수에 대한 연례 투표를 시작할 것을 요구합니다.
    "Shareholders ask Meta to initiate an annual vote on Meta executive pay, called say on pay."
P04-S02   p.70 | shareholder | opinion_commitment | 검증불가 | voting_governance
    2025 메타 연례 회의 투표에 있었던 것과 같은 임원 보수에 대한 투표가 매년 연례 회의 투표에 포함되도록 요구합니다.
    "the same vote on executive pay that was on the 2025 Meta annual meeting ballot, to be on each annual meeting ballot."
P04-S03   p.70 | shareholder | fact               | 검증가능 | voting_governance
    메타의 다음 임원 보수에 대한 주주 투표는 2028년까지 이루어지지 않을 것입니다.
    "the next Meta shareholder vote on executive pay will not be until 2028."
P04-S04   p.70 | shareholder | fact               | 검증가능 | voting_governance
    마크 주커버그는 메타에서 60%의 투표 권한을 가지고 있으며, 그는 자신의 모든 주식을 임원 보수 승인을 위해 투표합니다.
    "Mr. Zuckerberg has 60% voting power at Meta and Mr. Zuckerberg votes all his shares to approve executive pay."
P04-S05   p.70 |

In [ ]:
n = len(all_claims) or 1
print(f"원문 확인: {sum(c['quote_found'] for c in all_claims)}/{n}")
print(f"검증 가능: {sum(c['checkable'] for c in all_claims)}/{n}")
print(f"회사 요약 bullet 주장(★): {sum(c['in_board_summary'] for c in all_claims)}")

원문 확인: 17/17
검증 가능: 10/17
회사 요약 bullet 주장(★): 3


In [14]:
import pandas as pd

df = pd.DataFrame(all_claims)
df["★"] = df["in_board_summary"].map({True: "★", False: ""})
df["원문"] = df["quote_found"].map({True: "✅", False: "⚠"})

cols = ["claim_id", "★", "page_no", "speaker", "kind", "checkable", "topic", "원문", "claim_ko", "quote"]
pd.set_option("display.max_colwidth", 120)   # 긴 문장이 너무 잘리지 않게
df[cols]

,claim_id,★,page_no,speaker,kind,checkable,topic,원문,claim_ko,quote
0,P04-S01,,70,shareholder,opinion_commitment,False,voting_governance,✅,주주들은 메타가 '연례 임금 투표'라는 이름으로 메타의 임원 보수에 대한 연례 투표를 시작할 것을 요구합니다.,"Shareholders ask Meta to initiate an annual vote on Meta executive pay, called say on pay."
1,P04-S02,,70,shareholder,opinion_commitment,False,voting_governance,✅,2025 메타 연례 회의 투표에 있었던 것과 같은 임원 보수에 대한 투표가 매년 연례 회의 투표에 포함되도록 요구합니다.,"the same vote on executive pay that was on the 2025 Meta annual meeting ballot, to be on each annual meeting ballot."
2,P04-S03,,70,shareholder,fact,True,voting_governance,✅,메타의 다음 임원 보수에 대한 주주 투표는 2028년까지 이루어지지 않을 것입니다.,the next Meta shareholder vote on executive pay will not be until 2028.
3,P04-S04,,70,shareholder,fact,True,voting_governance,✅,"마크 주커버그는 메타에서 60%의 투표 권한을 가지고 있으며, 그는 자신의 모든 주식을 임원 보수 승인을 위해 투표합니다.",Mr. Zuckerberg has 60% voting power at Meta and Mr. Zuckerberg votes all his shares to approve executive pay.
4,P04-S05,,70,shareholder,fact,True,voting_governance,✅,메타는 임원 보수에 대한 연례 투표를 채택하기 위해 주주 승인 필요하지 않습니다.,Meta does not need shareholder approval to adopt an annual vote regarding executive pay.
5,P04-S06,,70,shareholder,opinion_commitment,False,voting_governance,✅,"임원 보수에 대한 연례 투표가 있다면 메타의 주주, 애널리스트 및 비즈니스 언론이 비주커버그 주식이 메타 임원 보수를 승인 또는 거부하는지를 추적할 동기를 더 가지게 될 것입니다.","with an annual vote regarding Meta executive pay Meta shareholders, analyst and business press will have more of an ..."
6,P04-S07,,70,shareholder,fact,True,other,✅,이는 시가총액이 1.8 조 달러인 회사에 특히 중요합니다.,This is especially important at a company that has a market cap of $1.8 Trillion.
7,P04-S08,,70,shareholder,fact,True,voting_governance,✅,이 제안의 제안자는 적시에 메타에 2025년 규칙 14a-8 제안을 제출했지만 그것은 2025 메타 연례 회의 투표에 나타나지 않았습니다.,the proponent of this proposal timely submitted a 2025 rule 14a-8 proposal to Meta but it did not appear on the 2025...
8,P04-S09,,70,shareholder,opinion_commitment,False,voting_governance,✅,[메타 직원]이 2025년 제안이 2025 메타 연례 회의 투표에 나타나지 않게 하기 위해 부정한 수단을 사용했다고 믿어집니다.,It is believed that [Meta employee] used dishonest means to prevent the 2025 proposal from appearing on the 2025 Met...
9,P04-S10,,70,shareholder,fact,True,voting_governance,✅,이 믿음을 뒷받침하는 증거가 있습니다.,There is evidence for this belief.


## 6. 저장
JSON은 다음 노트북용, CSV는 엑셀로 열어서 직접 확인·정답 라벨 붙이기용이에요.

In [21]:
suffix = "" if TEST_NUMBERS is None else "_test"
jpath = CLAIM_DIR / f"{FILING}_claims{suffix}.json"
jpath.write_text(json.dumps({"ticker": TICKER, "filingDate": FILING, "model": MODEL,
                             "usage": total, "claims": all_claims}, ensure_ascii=False, indent=2), encoding="utf-8")

cpath = CLAIM_DIR / f"{FILING}_claims{suffix}.csv"
if all_claims:
    with cpath.open("w", newline="", encoding="utf-8-sig") as f:   # utf-8-sig: 엑셀에서 한글 안 깨지게
        w = csv.DictWriter(f, fieldnames=list(all_claims[0].keys()))
        w.writeheader()
        w.writerows(all_claims)
print(f"저장: {jpath}\n저장: {cpath}")

저장: data/claims/meta/2026-04-16_claims_test.json
저장: data/claims/meta/2026-04-16_claims_test.csv


## 다음 단계
4번 안건 결과(5번 셀)를 보내 주세요. 특히 아래가 제대로 나왔는지 봐요.
- 회사 주장 **"2025년 주총에서 3년 주기가 승인됐다"** → `fact`, 검증 가능, ★
- "우리는 3년 주기가 적절하다고 믿는다" → `opinion_commitment`, 검증 불가

괜찮으면 `TEST_NUMBERS = None`으로 전체를 돌리고, `05_esg_priority.ipynb`(ESG 태깅 + 우선순위)로 넘어가요.